In [ ]:
import os
import pickle as pkl

from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats
from pydeseq2.utils import load_example_data

from pathlib import Path

SAVE = False  # whether to save the outputs of this notebook

if SAVE:
    # Replace this with the path to directory where you would like results to be saved
    OUTPUT_PATH = "../output_files/synthetic_example"
    os.makedirs(OUTPUT_PATH, exist_ok=True)  # Create path if it doesn't exist


# data paths 
DATA = Path('data')
probe_1 = DATA / 'lsm1_clone1.txt'
probe_2 = DATA / 'lsm1_clone2.txt'

In [ ]:
import pandas as pd
from pathlib import Path

# 1. Definiujemy ścieżki do plików
DATA = Path('data')
files = {
    'clone1': DATA / 'lsm1_clone1.txt',
    'clone2': DATA / 'lsm1_clone2.txt'
}

# 2. Wczytujemy i łączymy pliki w jedną macierz (counts_df)
# Zakładamy, że pliki nie mają nagłówka i są rozdzielone tabulatorem/spacją
all_counts = {}

for sample_name, file_path in files.items():
    df = pd.read_csv(file_path, sep=r'\s+', header=None, names=['gene_id', sample_name])
    df = df.set_index('gene_id')
    all_counts[sample_name] = df[sample_name]

# Tworzymy finalny DataFrame: geny w kolumnach, próbki w wierszach (wymóg PyDESeq2)
counts_df = pd.DataFrame(all_counts).T

# 3. Tworzymy metadane (metadata)
# Musisz ręcznie określić, która próbka należy do jakiej grupy
metadata = pd.DataFrame({
    'condition': ['control', 'treatment'] # Dostosuj te etykiety do swoich danych
}, index=counts_df.index)

print("Macierz zliczeń (pierwsze 5 genów):")
print(counts_df.iloc[:, :5])
print("\nMetadane:")
print(metadata)

In [ ]:
# filtering
## remove NAN values
samples_to_keep = ~metadata.condition.isna()
counts_df = counts_df.loc[samples_to_keep]
metadata = metadata.loc[samples_to_keep]
## At least 10 reads total
genes_to_keep = counts_df.columns[counts_df.sum(axis=0) >= 10]
counts_df = counts_df[genes_to_keep]

In [ ]:
# *** baseline 
# preparation of dataset
inference = DefaultInference(n_cpus=8)
dds = DeseqDataSet(
    counts=counts_df,
    metadata=metadata,
    design="~condition",
    refit_cooks=True,
    inference=inference,
    # n_cpus=8, # n_cpus can be specified here or in the inference object
)

In [ ]:
dds = DeseqDataSet(
    counts=counts_df,
    metadata=metadata,
    design_factors="condition",  # w nowszych wersjach używa się design_factors zamiast design
    refit_cooks=True,
    inference=inference,
)

In [ ]:
dds.deseq2()

In [ ]:
dds

In [ ]:
dds.deseq2()


if SAVE:
    with open(os.path.join(OUTPUT_PATH, "dds.pkl"), "wb") as f:
        pkl.dump(dds, f)

In [ ]:
print(dds)

In [ ]:
print(dds.var["dispersions"])


Co zrobić, przeprowadzenie analizy różnicowej, otrzymać plik z analizy deseq, przyjąć wartości graniczne (p value albo poprawka) oraz wybrać taka grupę genów, które mają:
- obniżoną ekspresje
- podwyższoną ekspresje 

Potem przeprowadzic analize funkcjonalną
- (przeglądarka - gprofiler) ale nie tego
- z pythona - jak najwygodniej 


#TODO 
- porównać robienie tego w R i w Pythonie

S batchem a nie s runem ??? 
- bardzo cwanie można sobie wybrać procesory podłączone do karty graficznej (bo mamy )